In [1]:
import numpy as np
import scipy as sp
import xarray as xr
import seaborn as sea
import matplotlib.pyplot as plt

# Codebook 

**Contains:**

- Histograms of variables --> mean of a year of data
- Histograms of variables --> seasonal variations

In [2]:
norkyst_gen_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024'
#choose all
import glob
filenames = glob.glob(f'{norkyst_gen_path}/*.nc')
ds = xr.open_mfdataset(filenames, engine = 'netcdf4').isel(s_rho = -1, s_w = -1)

KeyboardInterrupt: 

In [19]:
ds_monthly = ds.resample(time = 'MS').mean().compute()

MemoryError: 

In [ ]:
yearly_avg = ds_monthly.resample(time = 'Y').mean().compute()

/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(


MemoryError: 

In [14]:
#check if code from summer project is adaptable for this use case
sea.set_theme(style="darkgrid")
def histograms_with_specified_layout(ds, type_of_avg, time_to_check = 1, save_path = 'figures/histogram.png'):
    """
    Definition: 

    Arguments:
    arg[1] : dataset - the inference dataset 
    arg[2] : time to check (int) - the timestep in the data would you like the check the variable distribution for

    Outputs: 
    Figure of the variable distributions 
    """

    variables = ['salinity', 'u_eastward', 'v_northward', 'temperature', 'zeta']
    data = {var : ds[var].isel(time = time_to_check).values.ravel() for var in variables}
    data = {var : values[~np.isnan(values)] for var, values in data.items()}

    fig = plt.figure(figsize = (12,7))
    titles = ['Salinity', 'U-Eastward', 'V-Northward', 'temperature', 'zeta']
    """ 
    Trying to make the plot more visually appealing witg gridspec, to elongate the bottom plots.
    """
    gs = fig.add_gridspec(2,6)
    axes = [fig.add_subplot(gs[0,0:2]), fig.add_subplot(gs[0,2:4]), fig.add_subplot(gs[0,4:6]), fig.add_subplot(gs[1,0:3]), fig.add_subplot(gs[1,3:6])]
    
    for i, (var, values) in enumerate(data.items()):
        row, col = divmod(i,3)
        sea.histplot(values, bins = 30, kde = True, ax = axes[i], stat='density', color = 'royalblue', edgecolor = 'darkolivegreen')
        axes[i].set_title(titles[i], fontsize = 15)
        axes[i].set_facecolor("#C4C6D5")
        #axes[1,2].axis('off')

    plt.suptitle(f'Histogram & Density curve for Norkyst variables - {type_of_avg} average')
    plt.tight_layout()
    fig.savefig(save_path)

In [17]:
histograms_with_specified_layout(yearly_avg, f'Yearly', time_to_check=0, save_path=f'figures/yearly_avg.png')

KeyboardInterrupt: 

In [ ]:
#Now for the seasonal histograms 
def seasons(ds):
    #First - create monthly means, by resampling the data to include monthly means gathered at the start of each month
    monthly_avg = ds.resample(time = 'MS').mean()
    #Then we make weights based on the number of days in the respective months
    month_length = monthly_avg.time.dt.days_in_month
    #Preview
    print(month_length)
    #Weights - tid per måned / summen av alle tider  
    weights = month_length.groupby('time.season') / month_length.groupby('time.season').sum()
    #testing 
    if np.testing.assert_allclose(weights.groupby('time.season').sum().values, np.ones(4)) != 1.0:
        raise ValueError(f'Weights are not properly defined - please ensure that the sum of weights for the seasons equal 1.0')
    #apply weights to dataset
    ds_weighted = (ds*weights).groupby('time.season').sum(dim = 'time')
    return ds_weighted 